# Demo: Plot Combined Manuscript Source Data

This notebook reads only `data/spacestream_manuscript_source_data.xlsx`. Each worksheet is one manuscript figure panel; panels with multiple source tables are parsed from stacked sections within that worksheet.


In [ ]:
from pathlib import Path
import os
import zipfile
import xml.etree.ElementTree as ET

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
import seaborn as sns

ROOT = Path.cwd()
if ROOT.name != "manuscript_figure_source_data":
    ROOT = Path("/oak/stanford/groups/kalanit/biac2/kgs/projects/Dawn/manuscript_figure_source_data")

WORKBOOK = ROOT / "data" / "spacestream_manuscript_source_data.xlsx"
OUT = ROOT / "outputs" / "combined_demo_plots"
OUT.mkdir(parents=True, exist_ok=True)

os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", str(ROOT / ".cache"))
(ROOT / ".matplotlib").mkdir(exist_ok=True)
(ROOT / ".cache").mkdir(exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 200,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
    "font.family": "sans-serif",
    "font.sans-serif": ["Helvetica", "Arial", "DejaVu Sans"],
    "pdf.use14corefonts": True,
    "ps.useafm": True,
    "svg.fonttype": "none",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})
sns.set_theme(style="ticks", rc={
    "font.family": "sans-serif",
    "font.sans-serif": ["Helvetica", "Arial", "DejaVu Sans"],
})

CORE_ROI_NAMES = ["Ventral", "Lateral", "Dorsal"]
ROI_ORDER = ["Dorsal", "Lateral", "Ventral"]
ROI_COLORS = ["#377E2C", "#1A1AAC", "#8C1A4C"]
STREAM_PALETTE = {"Ventral": ROI_COLORS[2], "Lateral": ROI_COLORS[1], "Dorsal": ROI_COLORS[0]}
STREAM_COLORS = {
    "Dorsal": "#006600",
    "Parietal": "#006600",
    "Lateral": "#4d7fff",
    "Ventral": "#990000",
}
STREAM_ORDER = ["Dorsal", "Lateral", "Ventral"]
LEGACY_STREAM_ORDER = ["Parietal", "Lateral", "Ventral"]
CONTRAST_ORDER = ["Places", "Bodies", "Faces"]
TDANN_COLORS = {"simCLR": "#720298", "self_supervised": "#720298", "supervised": "#CB6D4A"}


def savefig(fig, stem):
    png = OUT / f"{stem}.png"
    pdf = OUT / f"{stem}.pdf"
    fig.tight_layout()
    fig.savefig(png)
    fig.savefig(pdf)
    plt.close(fig)
    print(png.relative_to(ROOT))
    print(pdf.relative_to(ROOT))


def despine(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def clean_legend_to_streams(ax, stream_order):
    handles, labels = ax.get_legend_handles_labels()
    stream_handles = {}
    for handle, label in zip(handles, labels):
        if label in stream_order and label not in stream_handles:
            stream_handles[label] = handle
    if stream_handles:
        ax.legend(stream_handles.values(), stream_handles.keys(), frameon=False, title="Stream")
    else:
        ax.legend([], [], frameon=False)



## Load Combined Panel Workbook


In [ ]:
NS_MAIN = "{http://schemas.openxmlformats.org/spreadsheetml/2006/main}"
NS_REL = "{http://schemas.openxmlformats.org/package/2006/relationships}"
NS_DOC_REL = "{http://schemas.openxmlformats.org/officeDocument/2006/relationships}"


def _read_shared_strings(zf):
    if "xl/sharedStrings.xml" not in zf.namelist():
        return []
    root = ET.fromstring(zf.read("xl/sharedStrings.xml"))
    return ["".join(node.text or "" for node in item.findall(f".//{NS_MAIN}t")) for item in root.findall(f"{NS_MAIN}si")]


def _read_sheet_rows(zf, sheet_path, shared_strings):
    root = ET.fromstring(zf.read(sheet_path))
    rows = []
    for row in root.findall(f".//{NS_MAIN}row"):
        values = []
        expected_col = 1
        for cell in row.findall(f"{NS_MAIN}c"):
            ref = cell.attrib.get("r", "A1")
            col_idx = 0
            for ch in "".join(ch for ch in ref if ch.isalpha()):
                col_idx = col_idx * 26 + (ord(ch.upper()) - 64)
            while expected_col < col_idx:
                values.append("")
                expected_col += 1
            cell_type = cell.attrib.get("t")
            if cell_type == "inlineStr":
                text_node = cell.find(f"{NS_MAIN}is/{NS_MAIN}t")
                values.append(text_node.text if text_node is not None and text_node.text is not None else "")
            elif cell_type == "s":
                value_node = cell.find(f"{NS_MAIN}v")
                idx = int(value_node.text) if value_node is not None and value_node.text is not None else -1
                values.append(shared_strings[idx] if 0 <= idx < len(shared_strings) else "")
            else:
                value_node = cell.find(f"{NS_MAIN}v")
                values.append(value_node.text if value_node is not None and value_node.text is not None else "")
            expected_col += 1
        while values and values[-1] == "":
            values.pop()
        rows.append(values)
    return rows


def _numericize(df):
    out = df.copy()
    for col in out.columns:
        try:
            out[col] = pd.to_numeric(out[col])
        except (TypeError, ValueError):
            pass
    return out


def load_source_workbook(path):
    with zipfile.ZipFile(path) as zf:
        shared_strings = _read_shared_strings(zf)
        workbook_xml = ET.fromstring(zf.read("xl/workbook.xml"))
        rels_xml = ET.fromstring(zf.read("xl/_rels/workbook.xml.rels"))
        rel_targets = {rel.attrib["Id"]: rel.attrib["Target"] for rel in rels_xml.findall(f"{NS_REL}Relationship")}
        panel_rows = {}
        for sheet in workbook_xml.findall(f".//{NS_MAIN}sheet"):
            name = sheet.attrib["name"]
            rid = sheet.attrib[f"{NS_DOC_REL}id"]
            sheet_path = "xl/" + rel_targets[rid].lstrip("/")
            panel_rows[name] = _read_sheet_rows(zf, sheet_path, shared_strings)
    return panel_rows


def extract_panel_tables(panel_rows):
    panel_tables = {}
    tables = {}
    csvs = {}
    metadata_keys = {"figure_panel", "source_workbook", "source_table", "staging_csv", "rows"}
    for panel, rows in panel_rows.items():
        panel_tables[panel] = {}
        i = 0
        while i < len(rows):
            row = rows[i]
            if row and row[0] == "source_table":
                table_name = row[1]
                staging_csv = ""
                i += 1
                while i < len(rows):
                    row = rows[i]
                    if row and row[0] == "staging_csv":
                        staging_csv = row[1] if len(row) > 1 else ""
                    if row and row[0] not in metadata_keys:
                        break
                    i += 1
                if i >= len(rows):
                    break
                header = rows[i]
                data_rows = []
                i += 1
                while i < len(rows):
                    row = rows[i]
                    if row and row[0] == "source_table":
                        break
                    if row:
                        data_rows.append(row)
                    i += 1
                width = len(header)
                padded = [r + [""] * max(width - len(r), 0) for r in data_rows]
                df = _numericize(pd.DataFrame([r[:width] for r in padded], columns=header))
                panel_tables[panel][table_name] = df
                tables[table_name] = df
                if staging_csv:
                    csvs[Path(staging_csv).stem] = df
                continue
            i += 1
    return panel_tables, tables, csvs

panel_rows = load_source_workbook(WORKBOOK)
panel_tables, sheets, csvs = extract_panel_tables(panel_rows)
print(f"Loaded {len(panel_tables)} panel sheets from {WORKBOOK}")
print(f"Extracted {len(sheets)} source tables")
for panel, table_map in panel_tables.items():
    print(panel, sorted(table_map))



## Fig 1A: Mega Matrix And MDS

In [ ]:
coords = csvs["fig1a_mds_coordinates_random_state0"].copy()
mega_matrix = csvs["fig1a_mega_matrix"].to_numpy(dtype=float)

roi_colors = {
    "Early": "#a6a6a6",
    "Midventral": "#f4bdd8",
    "Midlateral": "#ccdaff",
    "Midparietal": "#b3ffc6",
    "Ventral": "#DC267F",
    "Lateral": "#4d7fff",
    "Parietal": "#006600",
}
markers = {"lh": "v", "rh": "o"}

fig, ax = plt.subplots(figsize=(10, 10))
for (roi, hemi), group in coords.groupby(["roi_raw", "hemi"]):
    ax.scatter(
        group["mds_dim1"],
        group["mds_dim2"],
        s=35 + 20 * group["subject"].astype(int),
        c=roi_colors[roi],
        marker=markers[hemi],
        edgecolors="white",
        linewidths=0.5,
        alpha=0.9,
        label=f"{roi}, {hemi}",
    )
ax.set_xlabel("MDS dimension 1")
ax.set_ylabel("MDS dimension 2")
ax.set_title("Fig 1A MDS from saved mega matrix")
ax.spines[["top", "right"]].set_visible(False)
savefig(fig, "fig1a_mds_source_style_demo")

fig, ax = plt.subplots(figsize=(8, 8))
im = ax.imshow(mega_matrix, cmap="magma", vmin=0, vmax=1)
ax.set_title("Fig 1A source mega matrix")
ax.set_xlabel("Matrix index")
ax.set_ylabel("Matrix index")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
savefig(fig, "fig1a_mega_matrix_demo")


## Fig 1B: Human fLoc Selectivity

In [ ]:
def plot_floc_bar_strip(subject_df, hemi_df, stem, title, hue_order=None):
    sns.set_theme(style="ticks", context="talk")
    hue_order = hue_order or ["Dorsal", "Lateral", "Ventral"]
    fig, ax = plt.subplots(figsize=(6, 6), facecolor="white")
    ax.set_facecolor("white")

    sns.barplot(
        data=subject_df,
        x="contrast",
        y="frac_t_above",
        hue="stream",
        order=CONTRAST_ORDER,
        hue_order=hue_order,
        palette=STREAM_PALETTE,
        ci="sd",
        capsize=0.08,
        errwidth=1.5,
        edgecolor="white",
        linewidth=4,
        ax=ax,
    )

    markers = {"lh": "o", "rh": "^"}
    for hemi, marker in markers.items():
        sns.stripplot(
            data=hemi_df[hemi_df["hemi"] == hemi],
            x="contrast",
            y="frac_t_above",
            hue="stream",
            order=CONTRAST_ORDER,
            hue_order=hue_order,
            dodge=True,
            jitter=0.15,
            marker=marker,
            size=4.5,
            edgecolor="white",
            linewidth=0.5,
            palette=STREAM_PALETTE,
            ax=ax,
        )

    clean_legend_to_streams(ax, hue_order)
    ax.spines["right"].set_visible(False)
    ax.spines["top"].set_visible(False)
    ax.set_xlabel("Contrast")
    ax.set_ylabel("Fraction t > 3")
    ax.set_title(title)
    ax.grid(False)
    ax.tick_params(colors="black", labelcolor="black")
    ax.spines["left"].set_color("black")
    ax.spines["bottom"].set_color("black")
    savefig(fig, stem)

plot_floc_bar_strip(
    csvs["fig1b_human_floc_selectivity_subject_bars"],
    csvs["fig1b_human_floc_selectivity_hemi_dots"],
    "fig1b_human_floc_selectivity_source_style_demo",
    "Fig 1B human fLoc selectivity",
    hue_order=["Dorsal", "Lateral", "Ventral"],
)


## Figs 3B/3C/4A/4B: Model Correspondence Panels

In [ ]:
FIG3_MODEL_ORDER = [
    "MB_RN50", "MB_RN50_v2", "MB_RN50_v3", "MB_RN18",
    "TDANN_Supervised_0.0", "TDANN_SimCLR_0.0", "TDANN_Supervised", "TDANN_SimCLR",
]
FIG3_MODEL_LABELS = ["MB v1", "MB v2", "MB v3", "MB v1", "Cat", "SimCLR", "TDANN Cat", "TDANN SimCLR"]
FIG3_IN_BAR_LABELS = ["50", "50", "50", "18", "18", "18", "18", "18"]
FIG3_XVALS = [1, 2, 3, 4.5, 5.5, 6.5, 8, 9]

FIG4_MODEL_ORDER = [
    "MB_RN50_detection", "MB_RN50_action", "MB_RN50_categorization",
    "MB_RN50_v2_detection", "MB_RN50_v2_clip", "MB_RN50_v2_categorization",
    "MB_RN50_v3_depth", "MB_RN50_v3_pose", "MB_RN50_v3_categorization",
    "MB_RN18_detection", "MB_RN18_action", "MB_RN18_categorization",
]
FIG4_MODEL_LABELS = ["SSD", "SlowFast", "ResNet", "DETR", "CLIP", "ConvNeXT-T", "DepthAnythingv2", "DEKR-Pose", "ResNet", "Faster R-CNN", "SlowFast", "ResNet"]
FIG4_IN_BAR_LABELS = ["50", "50", "50", "50", "50", "50", "50", "50", "50", "18", "18", "18"]
FIG4_XVALS = [1, 2, 3, 5, 6, 7, 9, 10, 11, 13, 14, 15]

MATLAB_BAR_COLORS_FIG3 = ["#808080", "#808080", "#808080", "#808080", "#cc3300", "#660091", "#cc3300", "#660091"]
MATLAB_BAR_ALPHAS_FIG3 = [0.35, 0.35, 0.35, 0.5, 0.5, 0.5, 0.75, 1.0]
MATLAB_BAR_COLORS_FIG4 = ["#80b380", "#8080b3", "#cc8066", "#80b380", "#6666e6", "#cc8066", "#66cc66", "#4d4de6", "#cc8066", "#80b380", "#8080b3", "#cc8066"]


def plot_matlab_like_three_roi(points, noise, model_order, model_labels, in_bar_labels, xvals, colors, alphas, stem, title, ylabel):
    noise_summary = noise.groupby("ROI")["result"].agg(["mean", "std"])
    ymax = float((noise_summary["mean"] + noise_summary["std"]).max() * 1.1)
    fig, axes = plt.subplots(1, 3, figsize=(16, 6), sharey=True)

    for ax, roi in zip(axes, ROI_ORDER):
        roi_points = points[points["ROIS"] == roi]
        band = noise_summary.loc[roi]
        ax.axhspan(band["mean"] - band["std"], band["mean"] + band["std"], color="0.9", alpha=1.0, zorder=0)
        means = roi_points.groupby("model_type")["result"].mean()
        for i, model in enumerate(model_order):
            mean = means.loc[model] if model in means.index else np.nan
            ax.bar(xvals[i], mean, width=0.95, color=colors[i], alpha=alphas[i], edgecolor="none", zorder=2)
            if np.isfinite(mean):
                ax.text(xvals[i], ymax * 0.04, in_bar_labels[i], rotation=90, color="0.82", fontsize=13, ha="center", va="bottom")
            model_rows = roi_points[roi_points["model_type"] == model]
            lh = model_rows[model_rows["hemi"] == "lh"]["result"]
            rh = model_rows[model_rows["hemi"] == "rh"]["result"]
            edge = {"Dorsal": "#003300", "Lateral": "#000033", "Ventral": "#330018"}[roi]
            ax.scatter(np.full(len(lh), xvals[i]), lh, marker="^", facecolors="none", edgecolors=edge, linewidths=1.4, zorder=3)
            ax.scatter(np.full(len(rh), xvals[i]), rh, marker="o", facecolors="none", edgecolors=edge, linewidths=1.4, zorder=3)
        ax.set_xticks(xvals)
        ax.set_xticklabels(model_labels, rotation=90, fontsize=10)
        ax.set_xlim(0, max(xvals) + 1.5)
        ax.set_ylim(0, ymax)
        ax.set_title(roi, fontsize=18)
        ax.spines["right"].set_visible(False)
        ax.spines["top"].set_visible(False)
        if ax is axes[0]:
            ax.set_ylabel(ylabel, fontsize=14)
        else:
            ax.tick_params(labelleft=False)
            ax.spines["left"].set_visible(False)
    fig.suptitle(title, fontsize=18)
    savefig(fig, stem)

plot_matlab_like_three_roi(csvs["fig3b_spatial_correspondence_points"], csvs["fig3b_spatial_correspondence_noise_ceiling"], FIG3_MODEL_ORDER, FIG3_MODEL_LABELS, FIG3_IN_BAR_LABELS, FIG3_XVALS, MATLAB_BAR_COLORS_FIG3, MATLAB_BAR_ALPHAS_FIG3, "fig3b_spatial_correspondence_source_style_demo", "Fig 3B spatial correspondence", "Distance similarity [r]")
plot_matlab_like_three_roi(csvs["fig3c_functional_correspondence_points"], csvs["fig3c_functional_correspondence_noise_ceiling"], FIG3_MODEL_ORDER, FIG3_MODEL_LABELS, FIG3_IN_BAR_LABELS, FIG3_XVALS, MATLAB_BAR_COLORS_FIG3, MATLAB_BAR_ALPHAS_FIG3, "fig3c_functional_correspondence_source_style_demo", "Fig 3C functional correspondence", "Functional similarity [r]")
plot_matlab_like_three_roi(csvs["fig4a_mb_task_stream_assignment_points"], csvs["fig4a_mb_task_stream_assignment_noise_ceiling"], FIG4_MODEL_ORDER, FIG4_MODEL_LABELS, FIG4_IN_BAR_LABELS, FIG4_XVALS, MATLAB_BAR_COLORS_FIG4, [0.5] * 12, "fig4a_mb_task_stream_assignment_source_style_demo", "Fig 4A MB task-stream assignment", "Task-stream assignment [%]")
plot_matlab_like_three_roi(csvs["fig4b_mb_task_functional_correspondence_points"], csvs["fig4b_mb_task_functional_correspondence_noise_ceiling"], FIG4_MODEL_ORDER, FIG4_MODEL_LABELS, FIG4_IN_BAR_LABELS, FIG4_XVALS, MATLAB_BAR_COLORS_FIG4, [0.5] * 12, "fig4b_mb_task_functional_correspondence_source_style_demo", "Fig 4B MB task functional correspondence", "Functional similarity [r]")


## Fig 5B: Spatial Weight Curves

In [ ]:
def plot_fig5b_source_style(points, bands, stem_prefix, title_prefix, ylim_top):
    sns.set_theme(style="ticks")
    line_kwargs = {"marker": ".", "markersize": 18, "lw": 1, "markeredgecolor": "w", "markeredgewidth": 1.5}
    version_order = [v for v in ["self_supervised", "supervised"] if v in set(points["version"])]
    label_map = {"self_supervised": "Self-supervised", "supervised": "Supervised"}

    for streamx, stream in enumerate(CORE_ROI_NAMES):
        fig, ax = plt.subplots(figsize=(3, 5))
        df_stream = points[points["stream"] == stream].rename(columns={"spatial_weight": "Spatial_Weight", "corr": "Corr", "version": "Version"})
        sns.lineplot(
            data=df_stream,
            x="Spatial_Weight",
            y="Corr",
            hue="Version",
            hue_order=version_order,
            palette=[TDANN_COLORS[v] for v in version_order],
            errorbar="se",
            ax=ax,
            **line_kwargs,
        )
        ax.set_xscale("symlog", linthresh=0.09)
        ax.set_xlim([-0.01, 50])
        ax.set_xticks([], minor=True)
        ax.set_xticks([0, 0.1, 0.25, 0.5, 1.25, 2.5, 25])
        ax.set_xticklabels([0, 0.1, "", "", 1.25, "", 25])
        band = bands[bands["stream"] == stream].iloc[0]
        ax.axhspan(band["mean"] - band["std"], band["mean"] + band["std"], xmin=0, xmax=1, color="lightgray", alpha=0.75)
        if streamx == 2:
            h, _ = ax.get_legend_handles_labels()
            ax.legend(h, [label_map[v] for v in version_order], frameon=False)
        else:
            ax.legend([], [], frameon=False)
        ax.set_xlabel("Spatial Weight", fontsize=14)
        ax.set_ylabel("Correlation", fontsize=14)
        ax.set_yticks([])
        ax.spines["left"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.spines["top"].set_visible(False)
        ax.set_ylim(bottom=0.0, top=ylim_top)
        ax.set_title(f"{title_prefix}: {stream}")
        savefig(fig, f"{stem_prefix}_{stream.lower()}_source_style_demo")

plot_fig5b_source_style(csvs["fig5b_sc_spatial_correspondence_by_weight_points"], csvs["fig5b_sc_spatial_correspondence_brain_band_mean_sd"], "fig5b_top_spatial_correspondence", "Fig 5B top", 0.3)
plot_fig5b_source_style(csvs["fig5b_sc_functional_correspondence_by_weight_points"], csvs["fig5b_sc_functional_correspondence_brain_band_mean_sd"], "fig5b_bottom_functional_correspondence", "Fig 5B bottom", 0.5)


## Fig 5C: Effective Dimensionality vs Functional Similarity

In [ ]:
points = csvs["fig5c_effective_dimensionality_vs_functional_similarity_points"].copy()
points = points.rename(columns={"spatial_weight": "Spatial Weight", "type": "Type", "combined": "Combined", "stream": "Stream", "ed": "ED", "corr": "Corr"})
bands = csvs["fig5c_brain_reference_bands_mean_sd"]

for ridx, roi in enumerate(CORE_ROI_NAMES):
    fig, ax = plt.subplots(figsize=(7, 9))
    sns.set_theme(style="ticks")
    df_roi = points[points["Stream"] == roi]
    band = bands[bands["stream"] == roi].iloc[0]
    ax.axhspan(band["mean_corr"] - band["std_corr"], band["mean_corr"] + band["std_corr"], xmin=0, xmax=1, color="lightgray", alpha=0.75)
    ax.axvspan(band["mean_ed"] - band["std_ed"], band["mean_ed"] + band["std_ed"], color="lightgray", alpha=0.75)

    sup_open = df_roi[(df_roi["Type"] == "supervised") & ~(df_roi["Spatial Weight"].astype(str).isin(["2.5"]))]
    sim_open = df_roi[(df_roi["Type"] == "simCLR") & ~(df_roi["Spatial Weight"].astype(str).isin(["0.25", "0.5"]))]
    sup_filled = df_roi[(df_roi["Type"] == "supervised") & (df_roi["Spatial Weight"].astype(str).isin(["2.5"]))]
    sim_filled = df_roi[(df_roi["Type"] == "simCLR") & (df_roi["Spatial Weight"].astype(str).isin(["0.25", "0.5"]))]

    ax.scatter(sup_open["ED"], sup_open["Corr"], s=70, facecolors="white", edgecolors="#CB6D4A", linewidths=2, alpha=0.3, zorder=10, label="supervised")
    ax.scatter(sim_open["ED"], sim_open["Corr"], s=70, facecolors="white", edgecolors="#720298", linewidths=2, alpha=0.3, zorder=10, label="simCLR")
    ax.scatter(sup_filled["ED"], sup_filled["Corr"], s=130, color="#CB6D4A", alpha=0.65, zorder=11)
    ax.scatter(sim_filled["ED"], sim_filled["Corr"], s=90, color="#720298", alpha=0.65, zorder=11)
    ax.set_xscale("log")
    ax.set_ylim(0, 0.43)
    ax.set_xlabel("Effective Dimensionality")
    ax.set_ylabel("Correlation")
    ax.set_title(f"Fig 5C {roi}")
    ax.legend(frameon=False)
    ax.spines["right"].set_visible(False)
    ax.spines["top"].set_visible(False)
    savefig(fig, f"fig5c_ed_vs_functional_similarity_{roi.lower()}_source_style_demo")


## Figs 6A/6B: Transfer Panels

In [ ]:
def plot_transfer_source_style(points, value_col, stem, title, ylim=None):
    sns.set_theme(style="ticks")
    fig, ax = plt.subplots(figsize=(3, 6))
    raw_streams = {"Ventral": "Ventral", "Dorsal": "Parietal"}
    wide = points.pivot_table(index="line_index", columns="stream_display", values=value_col, aggfunc="first")
    for _, row in wide.dropna(subset=["Dorsal", "Ventral"]).iterrows():
        ax.plot([0.025, 0.975], [row["Dorsal"], row["Ventral"]], c="k", alpha=0.15)

    for stream_display, xpos, color in [("Ventral", 1, "#8C1A4C"), ("Dorsal", 0, "#377E2C")]:
        for hemi, marker in [("lh", "^"), ("rh", "o")]:
            vals = points[(points["stream_display"] == stream_display) & (points["hemi"] == hemi)][value_col]
            ax.scatter(
                np.full(len(vals), xpos), vals, color=color, s=70, alpha=0.95,
                edgecolors="w", marker=marker, linewidths=0.5,
            )
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Dorsal", "Ventral"], fontsize=16)
    ax.set_xlim([-0.2, 1.2])
    if ylim:
        ax.set_ylim(ylim)
    ax.spines["right"].set_visible(False)
    ax.spines["top"].set_visible(False)
    ax.set_ylabel(value_col.replace("_", " "))
    ax.set_title(title)
    savefig(fig, stem)

plot_transfer_source_style(csvs["fig6a_object_position_transfer_points"], "accuracy_proportion", "fig6a_object_position_transfer_source_style_demo", "Fig 6A object position", ylim=(0.425, 0.535))
plot_transfer_source_style(csvs["fig6b_imagenet_category_transfer_points"], "max_accuracy_percent", "fig6b_imagenet_category_transfer_source_style_demo", "Fig 6B ImageNet category")


## Fig 6C: Model fLoc Selectivity

In [ ]:
plot_floc_bar_strip(
    csvs["fig6c_model_floc_selectivity_subject_bars"],
    csvs["fig6c_model_floc_selectivity_hemi_dots"],
    "fig6c_model_floc_selectivity_source_style_demo",
    "Fig 6C model fLoc selectivity",
    hue_order=["Dorsal", "Lateral", "Ventral"],
)


## Fig 6D: Face-Selective RF Eccentricity

In [ ]:
df = csvs["fig6d_face_selective_rf_eccentricity_plot_points"].copy()
sns.set_theme(style="ticks")
fig, ax = plt.subplots(figsize=(4, 12))
order = ["Lateral", "Ventral"]
palette = [ROI_COLORS[1], ROI_COLORS[2]]

sns.stripplot(
    x="stream", y="eccen", hue="stream", jitter=0.1, linewidth=0.75, edgecolor="w",
    palette=palette, data=df[(df["hemi"] == "rh") & (df["stream"].isin(order))],
    order=order, dodge=True, size=9, ax=ax,
)
sns.stripplot(
    x="stream", y="eccen", hue="stream", jitter=0.2, linewidth=1, edgecolor="w", marker="^",
    palette=palette, data=df[(df["hemi"] == "lh") & (df["stream"].isin(order))],
    order=order, dodge=True, size=9, ax=ax,
)
sns.violinplot(
    x="stream", y="eccen", hue="stream", fill=True, linewidth=3, inner="box",
    saturation=0.9, palette=palette, data=df[df["stream"].isin(order)], order=order,
    dodge=False, ax=ax,
)
for collection in ax.collections:
    collection.set_alpha(0.85)
ax.spines["right"].set_visible(False)
ax.spines["top"].set_visible(False)
ax.legend([], [], frameon=False)
ax.set_xlabel("")
ax.set_ylabel("RF eccentricity")
ax.set_title("Fig 6D face-selective RF eccentricity")
savefig(fig, "fig6d_face_selective_rf_eccentricity_source_style_demo")


## S2A/S2B: Stream Correlations And Cross-ROI Fits


In [ ]:
def plot_s2a(points, summary):
    data = points.copy()
    summary = summary.copy()
    pair_order = ["Ventral/Lateral", "Ventral/Parietal", "Lateral/Parietal"]
    comparison_order = ["Within", "Between"]
    colors = {"Within": "#bdbdbd", "Between": "#525252"}
    x = np.arange(len(pair_order))
    width = 0.34
    offsets = {"Within": -width / 2, "Between": width / 2}

    fig, ax = plt.subplots(figsize=(4.6, 3.4), facecolor="white")
    for comparison in comparison_order:
        vals = []
        errs = []
        for pair in pair_order:
            row = summary[(summary["stream_pair"] == pair) & (summary["comparison"] == comparison)].iloc[0]
            vals.append(float(row["mean_fisher_z"]))
            errs.append(float(row["sem_fisher_z"]))
        xpos = x + offsets[comparison]
        ax.bar(xpos, vals, yerr=errs, width=width, color=colors[comparison], edgecolor="black", linewidth=0.8, capsize=2, label=comparison)
        for idx, pair in enumerate(pair_order):
            subj_vals = data[(data["stream_pair"] == pair) & (data["comparison"] == comparison)]["fisher_z"].to_numpy(float)
            jitter = np.linspace(-0.06, 0.06, len(subj_vals)) if len(subj_vals) else []
            ax.scatter(np.full(len(subj_vals), xpos[idx]) + jitter, subj_vals, s=18, color="lightgray", edgecolor="black", linewidth=0.35, zorder=3)
    ax.set_xticks(x)
    ax.set_xticklabels(pair_order, rotation=25, ha="right")
    ax.set_ylabel("RSM correlation (Fisher z)")
    ax.legend(frameon=False)
    despine(ax)
    fig.tight_layout()
    savefig(fig, "s2a_within_between_stream_correlations_demo")


def plot_s2b(points, summary):
    data = points.copy()
    summary = summary.copy()
    roi_order = ["Ventral", "Lateral", "Parietal"]
    model_order = ["same_roi", "other_roi1", "other_roi2"]
    model_labels = {"same_roi": "Same ROI", "other_roi1": "Other ROI 1", "other_roi2": "Other ROI 2"}
    colors = {"same_roi": "#f0f0f0", "other_roi1": "#969696", "other_roi2": "#252525"}
    x = np.arange(len(roi_order))
    width = 0.22
    offsets = {"same_roi": -width, "other_roi1": 0, "other_roi2": width}

    fig, ax = plt.subplots(figsize=(4.5, 4.4), facecolor="white")
    for model in model_order:
        vals = []
        errs = []
        for roi in roi_order:
            row = summary[(summary["roi"] == roi) & (summary["Model"] == model)].iloc[0]
            vals.append(float(row["mean_corrected"]))
            errs.append(float(row["sem_corrected"]))
        xpos = x + offsets[model]
        ax.bar(xpos, vals, yerr=errs, width=width, color=colors[model], edgecolor="black", linewidth=0.8, capsize=2, label=model_labels[model])
        for idx, roi in enumerate(roi_order):
            subj_vals = data[(data["roi"] == roi) & (data["Model"] == model)]["corrected"].to_numpy(float)
            jitter = np.linspace(-0.035, 0.035, len(subj_vals)) if len(subj_vals) else []
            ax.scatter(np.full(len(subj_vals), xpos[idx]) + jitter, subj_vals, s=18, color="lightgray", edgecolor="blue", linewidth=0.35, zorder=3)
    ax.axhline(1, color="black", linestyle="--", linewidth=1.0)
    ax.set_ylim(0, 1.05)
    ax.set_xticks(x)
    ax.set_xticklabels(roi_order, rotation=25, ha="right")
    ax.set_ylabel("Corrected R-squared")
    ax.legend(frameon=False, loc="upper right")
    despine(ax)
    fig.tight_layout()
    savefig(fig, "s2b_cross_roi_subject_to_subject_demo")

plot_s2a(sheets["s2a_points"], sheets["s2a_summary"])
plot_s2b(sheets["s2b_points"], sheets["s2b_summary"])



## S3C: Voxel-To-Voxel Accuracy, Checkpoint 0

In [ ]:
df = sheets["s3c_points"].copy()
df["display_roi"] = df["roi"].replace({"Parietal": "Dorsal"})
order = ["Dorsal", "Lateral", "Ventral"]

fig, ax = plt.subplots(figsize=(3.2, 4.2), facecolor="white")
x = np.arange(len(order))
for idx, roi in enumerate(order):
    vals = df.loc[df["display_roi"] == roi, "accuracy_percent"].to_numpy(float)
    ax.bar(idx, np.nanmean(vals), yerr=np.nanstd(vals, ddof=1), width=0.62,
           color=STREAM_COLORS[roi], alpha=0.75, edgecolor="white", linewidth=0.8)
    jitter = np.linspace(-0.12, 0.12, len(vals)) if len(vals) else []
    ax.scatter(np.full(len(vals), idx) + jitter, vals, s=26, color="lightgray", edgecolor="black", linewidth=0.4, zorder=3)
ax.axhline(33.3, color="black", linestyle="--", linewidth=1.0)
ax.set_xticks(x)
ax.set_xticklabels(order, rotation=30, ha="right")
ax.set_ylabel("Voxel assignment correspondence (%)")
ax.set_ylim(0, 100)
despine(ax)
fig.tight_layout()
savefig(fig, "s3c_voxel2voxel_accuracy_checkpoint0_demo")


## S5A/S5B: TDANN sw0.25 vs V1-Control

In [ ]:
def plot_s5_paired(points, stem, ylabel):
    data = points.copy()
    data["ROI"] = data["ROI"].replace({"Parietal": "Dorsal"})
    subj = data.groupby(["condition", "subject", "ROI"], as_index=False)["result"].mean()
    cond_order = ["TDANN_sw0.25", "TDANN_sw0.25_v1_control"]
    cond_labels = ["TDANN\nsw0.25", "V1-control\nsw0.25"]
    fig, axes = plt.subplots(1, 3, figsize=(7.2, 3.3), sharey=True, facecolor="white")
    for ax, roi in zip(axes, STREAM_ORDER):
        wide = subj[subj["ROI"] == roi].pivot(index="subject", columns="condition", values="result").dropna()
        vals = [wide[c].to_numpy(float) for c in cond_order]
        means = [np.nanmean(v) for v in vals]
        sems = [np.nanstd(v, ddof=1) / np.sqrt(np.sum(np.isfinite(v))) for v in vals]
        ax.bar([0, 1], means, yerr=sems, width=0.62, color=["#d7d7d7", "#9e9e9e"],
               edgecolor="#595959", linewidth=0.9, capsize=2, zorder=1)
        for _, row in wide.iterrows():
            ax.plot([0, 1], [row[cond_order[0]], row[cond_order[1]]], color="black", alpha=0.22, linewidth=0.7, zorder=2)
            ax.scatter([0, 1], [row[cond_order[0]], row[cond_order[1]]], color=STREAM_COLORS[roi],
                       edgecolor="white", linewidth=0.4, s=24, zorder=3)
        ax.set_title(roi)
        ax.set_xticks([0, 1])
        ax.set_xticklabels(cond_labels, rotation=25, ha="right")
        despine(ax)
    axes[0].set_ylabel(ylabel)
    fig.tight_layout()
    savefig(fig, stem)

plot_s5_paired(sheets["s5_spat_points"], "s5a_tdann_v1_control_spatial_demo", "Spatial similarity [r]")
plot_s5_paired(sheets["s5_func_points"], "s5b_tdann_v1_control_functional_demo", "Noise-corrected correlation")


## S6A/S6B: ViT-Control Correspondence Panels

In [ ]:
S6A_ORDER = [
    "MB_RN50_v2_detection",
    "MB_RN50_v2_vit_control_detection",
    "MB_RN50_v2_clip",
    "MB_RN50_v2_vit_control_clip",
    "MB_RN50_v2_categorization",
    "MB_RN50_v2_vit_control_categorization",
]
S6B_ORDER = [
    "MB_RN50_v2_detection",
    "MB_RN50_v2_detection_vit_control",
    "MB_RN50_v2_clip",
    "MB_RN50_v2_clip_vit_control",
    "MB_RN50_v2_categorization",
    "MB_RN50_v2_categorization_vit_control",
]
MODEL_LABELS = {
    "MB_RN50_v2_detection": "Det.",
    "MB_RN50_v2_vit_control_detection": "Det.\nV1",
    "MB_RN50_v2_detection_vit_control": "Det.\nV1",
    "MB_RN50_v2_clip": "CLIP",
    "MB_RN50_v2_vit_control_clip": "CLIP\nV1",
    "MB_RN50_v2_clip_vit_control": "CLIP\nV1",
    "MB_RN50_v2_categorization": "Cat.",
    "MB_RN50_v2_vit_control_categorization": "Cat.\nV1",
    "MB_RN50_v2_categorization_vit_control": "Cat.\nV1",
}
MODEL_COLORS = ["#80b380", "#80b380", "#6666cc", "#6666cc", "#cc8066", "#cc8066"]
MODEL_ALPHA = [1.0, 0.48, 1.0, 0.48, 1.0, 0.48]


def plot_s6(points, noise, model_order, stem, ylabel, chance=None, ylim=None):
    data = points.copy()
    data["ROIS"] = data["ROIS"].replace({"Parietal": "Dorsal"})
    noise = noise.copy()
    noise["ROI"] = noise["ROI"].replace({"Parietal": "Dorsal"})
    fig, axes = plt.subplots(1, 3, figsize=(9.0, 3.2), sharey=True, facecolor="white")
    x = np.array([0, 1, 2.15, 3.15, 4.3, 5.3])
    for ax, roi in zip(axes, STREAM_ORDER):
        sub = data[data["ROIS"] == roi]
        for idx, model in enumerate(model_order):
            vals = sub.loc[sub["model_type"] == model, "result"].to_numpy(float)
            ax.bar(x[idx], np.nanmean(vals), yerr=np.nanstd(vals, ddof=1), width=0.82,
                   color=MODEL_COLORS[idx], alpha=MODEL_ALPHA[idx], edgecolor="none", capsize=1.8, zorder=1)
            lh = sub[(sub["model_type"] == model) & (sub["hemi"] == "lh")]["result"].to_numpy(float)
            rh = sub[(sub["model_type"] == model) & (sub["hemi"] == "rh")]["result"].to_numpy(float)
            ax.scatter(np.full(len(lh), x[idx]) - 0.08, lh, marker="^", s=11, color="none", edgecolor="black", linewidth=0.35, zorder=3)
            ax.scatter(np.full(len(rh), x[idx]) + 0.08, rh, marker="o", s=11, color="none", edgecolor="black", linewidth=0.35, zorder=3)
        nvals = noise.loc[noise["ROI"] == roi, "result"].to_numpy(float)
        if len(nvals):
            ax.axhspan(np.nanmean(nvals) - np.nanstd(nvals, ddof=1), np.nanmean(nvals) + np.nanstd(nvals, ddof=1),
                       color="lightgray", alpha=0.35, zorder=0)
        if chance is not None:
            ax.axhline(chance, color="black", linestyle=":", linewidth=1.0)
        ax.set_title(roi)
        ax.set_xticks(x)
        ax.set_xticklabels([MODEL_LABELS[m] for m in model_order], rotation=0)
        if ylim:
            ax.set_ylim(*ylim)
        despine(ax)
    axes[0].set_ylabel(ylabel)
    fig.tight_layout()
    savefig(fig, stem)

plot_s6(sheets["s6a_points"], sheets["s6a_noise"], S6A_ORDER, "s6a_vit_control_stream_assignment_demo", "Stream assignment (%)", chance=33, ylim=(0, 100))
plot_s6(sheets["s6b_points"], sheets["s6b_noise"], S6B_ORDER, "s6b_vit_control_functional_correspondence_demo", "Functional correspondence [r]", ylim=(0, 0.36))


## S10: TDANN CKA Across Spatial Weights

In [ ]:
summary = sheets["s10_summary"].copy()
layer_order = [
    "base_model.maxpool", "base_model.layer1.0", "base_model.layer1.1",
    "base_model.layer2.0", "base_model.layer2.1", "base_model.layer3.0", "base_model.layer3.1",
    "base_model.layer4.0", "base_model.layer4.1",
]
pretty = [l.replace("base_model.", "").replace("layer", "layer ") for l in layer_order]
ceiling = summary[summary["comparison"] == "ceiling pooled (sw0.0 + sw0.25)"].set_index("layer").reindex(layer_order)
cross = summary[summary["comparison"] == "cross-weight (sw0.0 vs sw0.25)"].set_index("layer").reindex(layer_order)
x = np.arange(len(layer_order))

fig, ax = plt.subplots(figsize=(5.7, 2.8), facecolor="white")
c_mean = ceiling["mean_cka"].to_numpy(float)
c_sem = ceiling["sem_cka"].to_numpy(float)
x_mean = cross["mean_cka"].to_numpy(float)
ax.plot(x, c_mean, color="#2F6DAE", lw=1.4, label="Noise ceiling")
ax.fill_between(x, c_mean - c_sem, c_mean + c_sem, color="#2F6DAE", alpha=0.18, linewidth=0)
ax.plot(x, x_mean, color="#C55A11", marker="o", ms=4, lw=2.0, label="Cross-weight")
ax.set_xticks(x)
ax.set_xticklabels(pretty, rotation=35, ha="right")
ax.set_ylabel("Linear CKA")
ax.set_xlabel("TDANN layer")
ax.set_ylim(0, 1)
ax.grid(axis="y", color="#d9d9d9", linewidth=0.7, alpha=0.7)
ax.legend(frameon=False, loc="lower left")
despine(ax)
fig.tight_layout()
savefig(fig, "s10_cka_tdann_spatial_weight_demo")


## S12A: Functional Correspondence Across Spatial Weights

In [ ]:
def plot_s12a(points, bands):
    data = points.copy()
    data["stream"] = data["stream"].replace({"Parietal": "Dorsal"})
    bands = bands.copy()
    bands["stream"] = bands["stream"].replace({"Parietal": "Dorsal"})
    data["spatial_weight"] = pd.to_numeric(data["spatial_weight"])
    versions = [v for v in ["self_supervised", "supervised"] if v in set(data["version"])]
    palette = {"self_supervised": "#720298", "supervised": "#B59410"}
    labels = {"self_supervised": "SimCLR", "supervised": "Supervised"}
    fig, axes = plt.subplots(1, 3, figsize=(8.4, 3.0), sharey=True, facecolor="white")
    for ax, stream in zip(axes, STREAM_ORDER):
        sub = data[data["stream"] == stream]
        for version in versions:
            vsub = sub[sub["version"] == version]
            grouped = vsub.groupby("spatial_weight")["corr"].agg(["mean", "std", "count"]).reset_index().sort_values("spatial_weight")
            grouped["sem"] = grouped["std"] / np.sqrt(grouped["count"])
            ax.plot(grouped["spatial_weight"], grouped["mean"], marker=".", markersize=9, lw=1.5,
                    color=palette.get(version, "black"), label=labels.get(version, version))
            ax.fill_between(grouped["spatial_weight"].to_numpy(float),
                            (grouped["mean"] - grouped["sem"]).to_numpy(float),
                            (grouped["mean"] + grouped["sem"]).to_numpy(float),
                            color=palette.get(version, "black"), alpha=0.12, linewidth=0)
        band = bands[bands["stream"] == stream]
        if len(band):
            mean = float(band["mean"].iloc[0])
            sd = float(band["std"].iloc[0])
            ax.axhspan(mean - sd, mean + sd, color="lightgray", alpha=0.45, zorder=0)
        ax.set_xscale("symlog", linthresh=0.09)
        ax.set_xlim(-0.01, 50)
        ax.set_xticks([0, 0.1, 0.25, 0.5, 1.25, 2.5, 25])
        ax.set_xticklabels(["0", "0.1", "", "", "1.25", "", "25"])
        ax.set_title(stream)
        ax.set_xlabel("Spatial weight")
        despine(ax)
    axes[0].set_ylabel("Functional correspondence [r]")
    axes[-1].legend(frameon=False, loc="upper right")
    fig.tight_layout()
    savefig(fig, "s12a_functional_correspondence_by_weight_demo")

plot_s12a(sheets["s12a_points"], sheets["s12a_brain_band"])


## S11: HVM Transfer, Selected Tasks


In [ ]:
def plot_s11(seedavg):
    data = seedavg.copy()
    data["spatial_weight"] = pd.to_numeric(data["spatial_weight"])
    task_order = ["categorization", "combined_position", "rotation_xz", "size"]
    task_titles = {
        "categorization": "Categorization",
        "combined_position": "Position",
        "rotation_xz": "Rotation XZ",
        "size": "Size",
    }
    ylabels = {
        "categorization": "ImageNet accuracy (%)",
        "combined_position": "Position accuracy",
        "rotation_xz": "Rotation correlation [r]",
        "size": "Size correlation [r]",
    }
    fig, axes = plt.subplots(1, 4, figsize=(11.5, 3.0), facecolor="white")
    for ax, task in zip(axes, task_order):
        sub = data[data["task"] == task]
        for stream in STREAM_ORDER:
            grouped = (
                sub[sub["stream"] == stream]
                .groupby("spatial_weight")["result"]
                .agg(["mean", "std", "count"])
                .reset_index()
                .sort_values("spatial_weight")
            )
            grouped["sem"] = grouped["std"] / np.sqrt(grouped["count"])
            ax.plot(
                grouped["spatial_weight"],
                grouped["mean"],
                marker=".",
                markersize=9,
                lw=1.6,
                color=STREAM_COLORS[stream],
                label=stream,
            )
            ax.fill_between(
                grouped["spatial_weight"].to_numpy(float),
                (grouped["mean"] - grouped["sem"]).to_numpy(float),
                (grouped["mean"] + grouped["sem"]).to_numpy(float),
                color=STREAM_COLORS[stream],
                alpha=0.12,
                linewidth=0,
            )
        ax.set_xscale("symlog", linthresh=0.09)
        ax.set_xlim(-0.01, 50)
        ax.set_xticks([0, 0.1, 0.25, 0.5, 1.25, 2.5, 25])
        ax.set_xticklabels(["0", "0.1", "", "", "1.25", "", "25"])
        ax.set_title(task_titles[task])
        ax.set_xlabel("Spatial weight")
        ax.set_ylabel(ylabels[task])
        despine(ax)
    axes[-1].legend(frameon=False, loc="best")
    fig.tight_layout()
    savefig(fig, "s11_hvm_transfer_selected_tasks_demo")

plot_s11(sheets["s11_seedavg"])




## S12B: Supervised vs SimCLR Across Spatial Weights


In [ ]:
def plot_s12b(points, bands):
    data = points.copy()
    data["spatial_weight"] = pd.to_numeric(data["spatial_weight"])
    bands = bands.copy()
    palette = {"simCLR": "#720298", "supervised": "#B59410"}
    labels = {"simCLR": "SimCLR", "supervised": "Supervised"}
    versions = [v for v in ["simCLR", "supervised"] if v in set(data["version"])]
    fig, axes = plt.subplots(1, 3, figsize=(8.4, 3.0), sharey=True, facecolor="white")
    for ax, stream in zip(axes, STREAM_ORDER):
        sub = data[data["stream"] == stream]
        for version in versions:
            grouped = (
                sub[sub["version"] == version]
                .groupby("spatial_weight")["means"]
                .agg(["mean", "std", "count"])
                .reset_index()
                .sort_values("spatial_weight")
            )
            grouped["sem"] = grouped["std"] / np.sqrt(grouped["count"])
            ax.plot(
                grouped["spatial_weight"],
                grouped["mean"],
                marker=".",
                markersize=9,
                lw=1.5,
                color=palette[version],
                label=labels[version],
            )
            ax.fill_between(
                grouped["spatial_weight"].to_numpy(float),
                (grouped["mean"] - grouped["sem"]).to_numpy(float),
                (grouped["mean"] + grouped["sem"]).to_numpy(float),
                color=palette[version],
                alpha=0.12,
                linewidth=0,
            )
        band = bands[bands["stream"] == stream]
        if len(band):
            mean = float(band["mean"].iloc[0])
            sd = float(band["std"].iloc[0])
            ax.axhspan(mean - sd, mean + sd, color="lightgray", alpha=0.45, zorder=0)
        ax.set_xscale("symlog", linthresh=0.09)
        ax.set_xlim(-0.01, 50)
        ax.set_xticks([0, 0.1, 0.25, 0.5, 1.25, 2.5, 25])
        ax.set_xticklabels(["0", "0.1", "", "", "1.25", "", "25"])
        ax.set_title(stream)
        ax.set_xlabel("Spatial weight")
        despine(ax)
    axes[0].set_ylabel("Noise-corrected correlation")
    axes[-1].legend(frameon=False, loc="upper right")
    fig.tight_layout()
    savefig(fig, "s12b_supervised_vs_simclr_by_weight_demo")

plot_s12b(sheets["s12b_points"], sheets["s12b_brain_band"])



In [ ]:
print(f"Wrote combined demo plots to {OUT}")
print("PNG files:", len(list(OUT.glob("*.png"))))
print("PDF files:", len(list(OUT.glob("*.pdf"))))
